# 2199. Finding the Topic of Each Post

## Question
Leetcode has collected some posts from its social media website and is interested in finding the topics of each post.  

- Each topic can be expressed by one or more keywords.  
- If a keyword of a certain topic exists in the content of a post (**case insensitive**) then the post has this topic.  
- If the post does not have keywords from any topic, its topic should be **"Ambiguous!"**.  
- If the post has at least one keyword of any topic, its topic should be a string of the IDs of its topics sorted in ascending order and separated by commas `,`.  
- The string should not contain duplicate IDs.  

Return the result table in any order.

---

## Schema

### Table: Keywords
| Column Name | Type    | Description                                  |
|-------------|---------|----------------------------------------------|
| topic_id    | INT     | ID of the topic                              |
| word        | STRING  | Keyword that expresses the topic             |

**Primary Key:** (topic_id, word)

---

### Table: Posts
| Column Name | Type    | Description                                  |
|-------------|---------|----------------------------------------------|
| post_id     | INT     | Primary key, ID of the post                  |
| content     | STRING  | Content of the post (English letters/spaces) |

---

## Sample Data

### Keywords
| topic_id | word     |
|----------|----------|
| 1        | handball |
| 1        | football |
| 3        | WAR      |
| 2        | Vaccine  |

### Posts
| post_id | content                                                                |
|---------|------------------------------------------------------------------------|
| 1       | We call it soccer They call it football hahaha                         |
| 2       | Americans prefer basketball while Europeans love handball and football |
| 3       | stop the war and play handball                                         |
| 4       | warning I planted some flowers this morning and then got vaccinated    |

---



In [0]:
## PySpark Code to Create Schema, Data, and Temp Views


from pyspark.sql.types import StructType, StructField, IntegerType, StringType

# Schema for Keywords
keywords_schema = StructType([
    StructField("topic_id", IntegerType(), False),
    StructField("word", StringType(), False)
])

# Schema for Posts
posts_schema = StructType([
    StructField("post_id", IntegerType(), False),
    StructField("content", StringType(), False)
])

# Data for Keywords
keywords_data = [
    (1, "handball"),
    (1, "football"),
    (3, "WAR"),
    (2, "Vaccine")
]

# Data for Posts
posts_data = [
    (1, "We call it soccer They call it football hahaha"),
    (2, "Americans prefer basketball while Europeans love handball and football"),
    (3, "stop the war and play handball"),
    (4, "warning I planted some flowers this morning and then got vaccinated")
]

# Create DataFrames
keywords_df = spark.createDataFrame(keywords_data, keywords_schema)
posts_df = spark.createDataFrame(posts_data, posts_schema)

# Register Temp Views
keywords_df.createOrReplaceTempView("Keywords")
posts_df.createOrReplaceTempView("Posts")

# Quick check
keywords_df.show()
posts_df.show()


In [0]:
%sql
Select * from Posts p left join Keywords
on lower(p.content) like '%k.word%'

In [0]:
%sql
SELECT p.post_id, k.topic_id
FROM Posts p
JOIN Keywords k
  ON CHARINDEX(k.word, p.content) > 0;


In [0]:
%sql
SELECT p.post_id,
       COALESCE(
         STRING_AGG(DISTINCT k.topic_id, ',') WITHIN GROUP (ORDER BY k.topic_id),
         'Ambiguous!'
       ) AS topic
FROM Posts p
LEFT JOIN Keywords k
  ON LOWER(p.content) LIKE '%' + LOWER(k.word) + '%'
GROUP BY p.post_id;


In [0]:
%sql
SELECT p.post_id,
       COALESCE(
         ARRAY_JOIN(
           ARRAY_SORT(ARRAY_DISTINCT(COLLECT_LIST(k.topic_id))),
           ','
         ),
         'Ambiguous!'
       ) AS topic
FROM Posts p
LEFT JOIN Keywords k
  ON LOWER(p.content) LIKE CONCAT('%', LOWER(k.word), '%')
GROUP BY p.post_id;
